In [ ]:
%pip install pandas numpy scikit-learn nltk pyarrow fastparquet

In [2]:
import re
import nltk
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

In [4]:
# đọc dữ liệu từ tập train
train_df = pd.read_parquet('../data/train.parquet')
# đọc dữ liệu từ tập valid và test
test_df = pd.read_parquet('../data/test.parquet')
valid_df = pd.read_parquet('../data/valid.parquet')

# Prepare data

## Làm sạch dữ liệu

In [5]:
def create_brand(df):
  df['brand'] = df['details'].apply(lambda x: x.get('Brand') if isinstance(x, dict) else None)
  return df
def replace_missing_store_and_brand(df):
  df['store'] = df['store'].fillna('Unknown')
  df['brand'] = df['brand'].fillna('Unknown')
  return df

In [7]:
train_dfdf = create_brand(train_df)
train_df = replace_missing_store_and_brand(train_df)

valid_df = create_brand(valid_df)
valid_df = replace_missing_store_and_brand(valid_df)

test_df = create_brand(test_df)
test_df = replace_missing_store_and_brand(test_df)

In [8]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5220 entries, 0 to 5219
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             5220 non-null   int64         
 1   title_x            5220 non-null   str           
 2   text               5220 non-null   str           
 3   images_x           5220 non-null   object        
 4   asin               5220 non-null   str           
 5   parent_asin        5220 non-null   str           
 6   user_id            5220 non-null   str           
 7   timestamp          5220 non-null   datetime64[ns]
 8   helpful_vote       5220 non-null   int64         
 9   verified_purchase  5220 non-null   bool          
 10  main_category      5220 non-null   str           
 11  title_y            5220 non-null   str           
 12  average_rating     5220 non-null   float64       
 13  rating_number      5220 non-null   int64         
 14  features           

## Chọn lọc đặc trưng


In [10]:
def select_columns(train_df):
  cols_to_keep = [
    'user_id',
    'parent_asin',
    'rating',
    'helpful_vote',
    'brand',
    'text',
    'title_x',
    'title_y',
    'average_rating',
    'rating_number',
    'store'
   ]

  existing_cols = [col for col in cols_to_keep if col in train_df.columns]
  train_df = train_df[existing_cols].copy()
  return train_df

In [11]:
train_df = select_columns(  train_df )
print(train_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 5220 entries, 0 to 5219
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         5220 non-null   str    
 1   parent_asin     5220 non-null   str    
 2   rating          5220 non-null   int64  
 3   helpful_vote    5220 non-null   int64  
 4   brand           5220 non-null   str    
 5   text            5220 non-null   str    
 6   title_x         5220 non-null   str    
 7   title_y         5220 non-null   str    
 8   average_rating  5220 non-null   float64
 9   rating_number   5220 non-null   int64  
 10  store           5220 non-null   str    
dtypes: float64(1), int64(3), str(7)
memory usage: 4.5 MB
None


## Chỉnh sửa dữ liệu

In [12]:
def transform_rating_into_liked(df, threshold=4):
  df['liked'] = (df['rating'] >= 4).astype(int)
  return df

In [13]:
train_df = transform_rating_into_liked(train_df)
test_df = transform_rating_into_liked(test_df)
valid_df = transform_rating_into_liked(valid_df)

In [14]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5220 entries, 0 to 5219
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         5220 non-null   str    
 1   parent_asin     5220 non-null   str    
 2   rating          5220 non-null   int64  
 3   helpful_vote    5220 non-null   int64  
 4   brand           5220 non-null   str    
 5   text            5220 non-null   str    
 6   title_x         5220 non-null   str    
 7   title_y         5220 non-null   str    
 8   average_rating  5220 non-null   float64
 9   rating_number   5220 non-null   int64  
 10  store           5220 non-null   str    
 11  liked           5220 non-null   int64  
dtypes: float64(1), int64(4), str(7)
memory usage: 4.6 MB


In [19]:
def scale_numeric_features(train_df, valid_df, test_df ):
  numeric_cols = [ 'helpful_vote', 'average_rating', 'rating_number']

  scaler = StandardScaler()

  # Fit trên train, transform train
  train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])

  # Transform trên valid và test
  valid_df[numeric_cols] = scaler.transform(valid_df[numeric_cols])
  test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

  return train_df, valid_df, test_df

In [20]:
train_df , valid_df, test_df = scale_numeric_features(train_df , valid_df, test_df)

In [22]:
def target_encode(train, val, test, col, target='liked', alpha=10):
    """
    Target encoding với smoothing cho biến categorical.
    - train: DataFrame train
    - val: DataFrame validation (có thể None nếu không có)
    - test: DataFrame test
    - col: tên cột cần encode
    - target: tên cột target (0/1)
    - alpha: hệ số smoothing (càng lớn càng kéo về global mean)
    """
    global_mean = train[target].mean()

    # Tính mean target và count cho từng giá trị trong train
    stats = train.groupby(col)[target].agg(['mean', 'count']).reset_index()
    stats.columns = [col, 'mean_target', 'count']

    # Smoothing
    stats['encoded'] = (stats['count'] * stats['mean_target'] + alpha * global_mean) / (stats['count'] + alpha)

    # Gắn vào train
    train = train.merge(stats[[col, 'encoded']], on=col, how='left')
    train[f'{col}_enc'] = train['encoded']
    train.drop(columns=['encoded'], inplace=True)

    # Gắn vào val (nếu có)
    if val is not None:
        val = val.merge(stats[[col, 'encoded']], on=col, how='left')
        val[f'{col}_enc'] = val['encoded'].fillna(global_mean)
        val.drop(columns=['encoded'], inplace=True)

    # Gắn vào test
    test = test.merge(stats[[col, 'encoded']], on=col, how='left')
    test[f'{col}_enc'] = test['encoded'].fillna(global_mean)
    test.drop(columns=['encoded'], inplace=True)

    return train, val, test



In [23]:
train_df, valid_df, test_df = target_encode(train_df, valid_df, test_df, 'brand', alpha=10)
train_df, valid_df, test_df = target_encode(train_df, valid_df, test_df, 'store', alpha=10)

In [24]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5220 entries, 0 to 5219
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         5220 non-null   str    
 1   parent_asin     5220 non-null   str    
 2   rating          5220 non-null   int64  
 3   helpful_vote    5220 non-null   float64
 4   brand           5220 non-null   str    
 5   text            5220 non-null   str    
 6   title_x         5220 non-null   str    
 7   title_y         5220 non-null   str    
 8   average_rating  5220 non-null   float64
 9   rating_number   5220 non-null   float64
 10  store           5220 non-null   str    
 11  liked           5220 non-null   int64  
 12  brand_enc       5220 non-null   float64
 13  store_enc       5220 non-null   float64
dtypes: float64(5), int64(2), str(7)
memory usage: 4.6 MB


In [25]:
# ---------------------------------
# Phần khởi tạo tài nguyên NLTK (chạy một lần khi import module)
# ---------------------------------
def _download_nltk_resources():
    """Tải các gói dữ liệu NLTK cần thiết (nội bộ)."""
    resources = ['punkt', 'punkt_tab', 'stopwords']
    for res in resources:
        try:
            if res.startswith('punkt'):
                nltk.data.find(f'tokenizers/{res}')
            else:
                nltk.data.find(f'corpora/{res}')
        except LookupError:
            nltk.download(res, quiet=True)

_download_nltk_resources()

# Các hằng số dùng chung (chỉ khởi tạo một lần)
_STOP_WORDS = set(stopwords.words('english'))
_STEMMER = PorterStemmer()

# ---------------------------------
# Các hàm tiền xử lý văn bản (nội bộ)
# ---------------------------------
def _clean_text(text):
    """Làm sạch văn bản: chữ thường, loại bỏ dấu câu, ký tự đặc biệt."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

def _tokenize_and_stem(text):
    """Tokenize, loại bỏ stop words & từ ngắn, sau đó stemming."""
    words = nltk.word_tokenize(text)
    words = [_STEMMER.stem(w) for w in words if w not in _STOP_WORDS and len(w) > 2]
    return ' '.join(words)

def _preprocess_series(series):
    """Áp dụng toàn bộ tiền xử lý lên một pandas Series."""
    return series.apply(_clean_text).apply(_tokenize_and_stem)

def _combine_text_columns(df, columns):
    """Ghép nội dung các cột văn bản thành một chuỗi (fillna bằng chuỗi rỗng)."""
    return df[columns].fillna('').agg(' '.join, axis=1)

# ---------------------------------
# Hàm pipeline chính (public)
# ---------------------------------
def nlp_pipeline(train_df, val_df, test_df, text_columns=['text'],
                 max_features=5000, ngram_range=(1,2)):
    """
    Xử lý NLP hoàn chỉnh cho train, validation, test với nhiều cột văn bản.

    Parameters:
    -----------
    train_df, val_df, test_df : pandas.DataFrame
        Các DataFrame có chứa các cột văn bản.
    text_columns : list
        Danh sách tên cột cần ghép (mặc định chỉ ['text']).
    max_features : int
        Số lượng đặc trưng tối đa cho TF-IDF.
    ngram_range : tuple
        Khoảng n-gram (ví dụ (1,2) cho unigram + bigram).

    Returns:
    --------
    X_train : sparse matrix
    X_val : sparse matrix
    X_test : sparse matrix
    vectorizer : TfidfVectorizer
    """
    # 1. Ghép các cột văn bản
    train_combined = _combine_text_columns(train_df, text_columns)
    val_combined = _combine_text_columns(val_df, text_columns)
    test_combined = _combine_text_columns(test_df, text_columns)

    # 2. Tiền xử lý
    train_clean = _preprocess_series(train_combined)
    val_clean = _preprocess_series(val_combined)
    test_clean = _preprocess_series(test_combined)

    # 3. TF-IDF (chỉ fit trên train)
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    X_train = vectorizer.fit_transform(train_clean)
    X_val = vectorizer.transform(val_clean)
    X_test = vectorizer.transform(test_clean)

    return X_train, X_val, X_test, vectorizer

In [28]:

X_train, X_valid, X_test, tfidf_vec = nlp_pipeline(
       train_df, valid_df, test_df,
       text_columns=['text', 'title_x', 'title_y'],
        max_features=5000
    )
print("Train shape:", X_train.shape)
print("Val shape:", X_valid.shape)
print("Test shape:", X_test.shape)

Train shape: (5220, 5000)
Val shape: (1214, 5000)
Test shape: (1214, 5000)


# Tổng kết

In [184]:
def Prepare_data(train_df, valid_df, test_df):
  # làm sạch dữ liệu missing trong brand và store
  train_df = create_brand(train_df)
  train_df = replace_missing_store_and_brand(train_df)

  valid_df = create_brand(valid_df)
  valid_df = replace_missing_store_and_brand(valid_df)

  test_df = create_brand(test_df)
  test_df = replace_missing_store_and_brand(test_df)

  # Chọn lọc đặc trưng cho tập train

  train_df = select_columns(train_df)

  # chỉnh sửa dữ liệu trên tập train, valid , test

  train_df = transform_rating_into_liked(train_df)
  test_df = transform_rating_into_liked(test_df)
  valid_df = transform_rating_into_liked(valid_df)

  train_df, valid_df, test_df = scale_numeric_features(train_df, valid_df, test_df)

  train_df, valid_df, test_df = target_encode(train_df, valid_df, test_df, 'brand', alpha=10)
  train_df, valid_df, test_df = target_encode(train_df, valid_df, test_df, 'store', alpha=10)

  return train_df, valid_df, test_df